<a href="https://colab.research.google.com/github/supunabeywickrama/my-colab-work/blob/main/video_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
!apt-get update -y
!apt-get install -y \
  libatk-bridge2.0-0 \
  libatk1.0-0 \
  libcups2 \
  libxcomposite1 \
  libxdamage1 \
  libxrandr2 \
  libgbm1 \
  libasound2 \
  libpangocairo-1.0-0 \
  libgtk-3-0 \
  libnss3 \
  libxshmfence1



Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,227 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [345 B]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,860 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all P

In [26]:
!pip install -U streamlit pyngrok playwright ffmpeg-python
!playwright install chromium


In [27]:
from pyngrok import ngrok

ngrok.set_auth_token("37s941nK8k6ssqu1GxpWVX1h6kJ_5sgAaJDixMGSnQoAspzNT")

In [28]:
%%writefile app.py
import streamlit as st
import asyncio
from playwright.async_api import async_playwright
import subprocess
import os
import time

st.set_page_config(page_title="Smart Video Downloader", layout="wide")
st.title("🎥 Smart Video Downloader")

url = st.text_input("Paste Website URL")

if not url:
    st.stop()

# ---------- Preview ----------
st.subheader("🌐 Website Preview (Play video here)")
st.components.v1.iframe(url, height=420, scrolling=True)

st.info("▶️ Play the video in the preview, then click **Detect Video**")

# ---------- Video Detection ----------
async def detect_video(page_url):
    found = set()

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"]
        )
        context = await browser.new_context()
        page = await context.new_page()

        page.on(
            "response",
            lambda r: (
                found.add(r.url)
                if (".m3u8" in r.url or ".mp4" in r.url)
                else None
            )
        )

        await page.goto(page_url, wait_until="domcontentloaded")
        await asyncio.sleep(8)  # allow user-played video to load

        await browser.close()

    return list(found)

# ---------- Detect Button ----------
if st.button("🔍 Detect Video"):
    with st.spinner("Listening for video streams (play video now if not already)..."):
        video_urls = asyncio.run(detect_video(url))

    if not video_urls:
        st.error("❌ Video is playing, but stream is protected or DRM-based.")
        st.info("This usually means encrypted HLS / tokenized / DRM content.")
        st.stop()

    main_video = video_urls[0]
    st.success("✅ Video stream detected!")

    st.video(main_video)

    # ---------- Download ----------
    if st.button("⬇️ Download Video"):
        with st.spinner("Downloading video..."):
            cmd = [
                "ffmpeg",
                "-headers", f"User-Agent: Mozilla/5.0\r\nReferer: {url}",
                "-i", main_video,
                "-c", "copy",
                "downloaded_video.mp4"
            ]
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        if os.path.exists("downloaded_video.mp4"):
            st.success("🎉 Download completed! Check Colab Files.")
        else:
            st.error("Download failed (encrypted or expired stream).")


Writing app.py


In [29]:
from pyngrok import ngrok
import os

ngrok.kill()
print("OPEN WEBSITE:", ngrok.connect(8501))
os.system("streamlit run app.py &")




OPEN WEBSITE: NgrokTunnel: "https://overempirical-unparticipative-yuonne.ngrok-free.dev" -> "http://localhost:8501"


0